In [ ]:
# ===========================================
# Notebook Name:
# 01_deck_similarity_explorer
#
# Purpose:
# Interactive exploration of gold.deck_similarity:
# given a deck_hash, list its most similar decks
# by weighted Pokemon-composition Jaccard
# similarity, alongside each deck's tournament
# performance from gold.deck_registry.
#
# This notebook is exploratory only and is NOT
# part of the Daily/Weekly Workflow.
#
# Input:
# pokemon.gold.deck_similarity
# pokemon.gold.deck_registry
# pokemon.gold.deck_pokemon_features
# ===========================================

In [ ]:
from pyspark.sql import functions as F

DECK_SIMILARITY_TABLE = "pokemon.gold.deck_similarity"
DECK_REGISTRY_TABLE = "pokemon.gold.deck_registry"
DECK_POKEMON_FEATURES_TABLE = "pokemon.gold.deck_pokemon_features"

deck_similarity_df = spark.table(DECK_SIMILARITY_TABLE)
deck_registry_df = spark.table(DECK_REGISTRY_TABLE)
deck_pokemon_features_df = spark.table(DECK_POKEMON_FEATURES_TABLE)

In [ ]:
dbutils.widgets.text("deck_hash", "")
dbutils.widgets.text("top_n", "10")

print(
    "Fill in the deck_hash widget with a "
    "gold.deck_registry.deck_hash to explore. "
    "Leave blank to default to the deck with the "
    "most tournament appearances."
)

In [ ]:
requested_deck_hash = (
    dbutils.widgets.get("deck_hash").strip()
)

requested_top_n = int(
    dbutils.widgets.get("top_n").strip() or "10"
)

if requested_deck_hash:
    target_deck_hash = requested_deck_hash
else:
    default_deck_row = (
        deck_registry_df
        .orderBy(F.col("tournament_count").desc())
        .select("deck_hash")
        .first()
    )

    if default_deck_row is None:
        raise ValueError(
            f"{DECK_REGISTRY_TABLE} is empty. "
            "Run 03_gold/01_build_deck_registry first."
        )

    target_deck_hash = default_deck_row["deck_hash"]

print("Target deck_hash:", target_deck_hash)
print("Top N:", requested_top_n)

In [ ]:
target_deck_row = (
    deck_registry_df
    .filter(F.col("deck_hash") == target_deck_hash)
    .first()
)

if target_deck_row is None:
    raise ValueError(
        f"deck_hash {target_deck_hash} not found in "
        f"{DECK_REGISTRY_TABLE}."
    )

print("Representative deck_code:", target_deck_row["representative_deck_code"])
print("Tournament appearances  :", target_deck_row["tournament_count"])
print("Best rank               :", target_deck_row["best_rank"])
print("Win rate                :", target_deck_row["win_rate_pct"])

In [ ]:
similar_decks_df = (
    deck_similarity_df
    .filter(
        (F.col("deck_hash_a") == target_deck_hash)
        | (F.col("deck_hash_b") == target_deck_hash)
    )
    .select(
        F.when(
            F.col("deck_hash_a") == target_deck_hash,
            F.col("deck_hash_b"),
        )
        .otherwise(F.col("deck_hash_a"))
        .alias("other_deck_hash"),
        "shared_pokemon_names",
        "union_pokemon_names",
        "total_quantity_difference",
        "weighted_jaccard_similarity",
        "weighted_jaccard_pct",
        "presence_jaccard_similarity",
        "presence_jaccard_pct",
    )
    .orderBy(F.col("weighted_jaccard_similarity").desc())
    .limit(requested_top_n)
)

similar_decks_with_registry_df = (
    similar_decks_df
    .join(
        deck_registry_df.select(
            F.col("deck_hash").alias("other_deck_hash"),
            F.col("representative_deck_code").alias("other_representative_deck_code"),
            F.col("tournament_count").alias("other_tournament_count"),
            F.col("best_rank").alias("other_best_rank"),
            F.col("win_rate_pct").alias("other_win_rate_pct"),
        ),
        on="other_deck_hash",
        how="left",
    )
    .orderBy(F.col("weighted_jaccard_similarity").desc())
)

display(similar_decks_with_registry_df)

In [ ]:
TOP_SHARED_POKEMON_LIMIT = 15

top_similar_deck_row = (
    similar_decks_with_registry_df.first()
)

if top_similar_deck_row is not None:
    comparison_deck_hash = (
        top_similar_deck_row["other_deck_hash"]
    )

    target_pokemon_df = (
        deck_pokemon_features_df
        .filter(
            (F.col("deck_hash") == target_deck_hash)
            & (F.col("is_present") == 1)
        )
        .select("card_name", "quantity")
    )

    comparison_pokemon_df = (
        deck_pokemon_features_df
        .filter(
            (F.col("deck_hash") == comparison_deck_hash)
            & (F.col("is_present") == 1)
        )
        .select("card_name", "quantity")
    )

    pokemon_comparison_df = (
        target_pokemon_df.alias("target")
        .join(
            comparison_pokemon_df.alias("comparison"),
            on="card_name",
            how="full_outer",
        )
        .select(
            "card_name",
            F.col("target.quantity").alias("target_quantity"),
            F.col("comparison.quantity").alias("comparison_quantity"),
        )
        .orderBy(
            F.coalesce(
                F.col("target_quantity"),
                F.lit(0),
            ).desc()
        )
        .limit(TOP_SHARED_POKEMON_LIMIT)
    )

    print(
        "Comparing target",
        target_deck_hash,
        "against most similar deck",
        comparison_deck_hash,
    )

    display(pokemon_comparison_df)
else:
    print(
        "No similar decks found for this deck_hash. "
        "gold.deck_similarity may not have been "
        "built yet, or this deck has no pairs above "
        "the similarity threshold used at build time."
    )